In [ ]:
"""
Toronto Paper Reproduction (v3 — final refinement)
===================================================
Paper: Prediction of Fatal and Major Injury of Drivers, Cyclists,
       and Pedestrians in Collisions (Shanshal et al., 2020)
       https://doi.org/10.7307/ptt.v32i1.3134

Fixes vs v2:
  - One-hot encoding for categoricals (Lasso needs it for sparsity)
  - Tune Lasso C via grid search + CV on training fold
  - Keep all INVTYPEs & all records → closer to 8,922 obs
  - 27 features matching paper's variable set
  - SMOTE applied correctly on training data only (no leakage)
  - Handle missing → 'Unknown' (not dropped), per paper's internal validity
"""

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE
from mlxtend.frequent_patterns import apriori, association_rules
from scipy.stats import spearmanr

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

OUTPUT_DIR = './outputs/toronto_paper_reproduction'
os.makedirs(OUTPUT_DIR, exist_ok=True)

ksi = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "ngducchilly/toronto-ksi-collisions-20062023",
    "TOTAL_KSI_3737821728629277523.csv",
)

In [ ]:
print("=" * 72)
print("PREDICTION OF FATAL AND MAJOR INJURY — Toronto Paper Reproduction v3")
print("Shanshal, Babaoglu, Basar (2020) — Promet 32(1), 39-53")
print("=" * 72)

In [ ]:
# ====================================================================
# 1. LOAD & FILTER
# ====================================================================
print("\n1. DATA LOADING", flush=True)
print(f"  Raw: {ksi.shape[0]:,d} rows", flush=True)

In [ ]:
ksi['DATE'] = pd.to_datetime(ksi['DATE'], errors='coerce')
ksi['YEAR'] = ksi['DATE'].dt.year
ksi = ksi[ksi['YEAR'].between(2007, 2017)].copy()
print(f"  2007-2017: {ksi.shape[0]:,d} rows", flush=True)

In [ ]:
if 'DIVISION' in ksi.columns and 'POLICE_DIVISION' not in ksi.columns:
    ksi.rename(columns={'DIVISION': 'POLICE_DIVISION'}, inplace=True)

In [ ]:
# ====================================================================
# 2. TARGET
# ====================================================================
print("\n2. TARGET", flush=True)
ksi['INJURY'] = ksi['INJURY'].fillna('None')
inj_counts = ksi['INJURY'].value_counts()
print(f"  INJURY: {inj_counts.to_dict()}", flush=True)
print(f"  Total: {inj_counts.sum():,d}  (paper: 8,922)", flush=True)

In [ ]:
ksi['target_binary'] = np.where(ksi['INJURY'].isin(['Fatal', 'Major']), 1, 0)
n_pos, n_neg = ksi['target_binary'].sum(), (1 - ksi['target_binary']).sum()
print(f"  Binary: 1={n_pos:,d}, 0={n_neg:,d}, ratio={n_pos/(n_pos+n_neg):.3f}", flush=True)

In [ ]:
# ====================================================================
# 3. FEATURE ENGINEERING
# ====================================================================
print("\n3. FEATURE ENGINEERING", flush=True)
ksi['Month'] = ksi['DATE'].dt.month.fillna(6).astype(int)

In [ ]:
def _hour(t):
    if pd.isna(t) or str(t).strip() == '': return 12
    try: return min(max(int(str(t).strip().zfill(4)[:2]), 0), 23)
    except: return 12
ksi['Hour'] = ksi['TIME'].apply(_hour)
ksi['DayOfWeek'] = ksi['DATE'].dt.dayofweek.fillna(0).astype(int)

In [ ]:
# Fill missing → 'Unknown' (paper approach: keep, don't drop)
cat_cols = ['IMPACTYPE', 'VISIBILITY', 'LIGHT', 'RDSFCOND', 'ROAD_CLASS',
            'TRAFFCTL', 'ACCLOC', 'DRIVACT', 'DRIVCOND', 'MANOEUVER',
            'VEHTYPE', 'INVTYPE', 'PEDESTRIAN', 'CYCLIST', 'AUTOMOBILE',
            'MOTORCYCLE', 'TRUCK', 'PASSENGER', 'SPEEDING', 'AG_DRIV',
            'REDLIGHT', 'ALCOHOL', 'DISABILITY', 'POLICE_DIVISION']
for c in cat_cols:
    if c in ksi.columns: ksi[c] = ksi[c].fillna('Unknown')

In [ ]:
invage = pd.to_numeric(ksi['INVAGE'], errors='coerce')
ksi['INVAGE'] = invage.fillna(invage.median())

In [ ]:
# ====================================================================
# 4. VARIABLE SELECTION → 27 variables (paper: 26)
# ====================================================================
print("\n4. VARIABLE SELECTION", flush=True)

In [ ]:
raw_features = ['IMPACTYPE', 'VISIBILITY', 'LIGHT', 'RDSFCOND', 'ROAD_CLASS',
                'TRAFFCTL', 'ACCLOC', 'DRIVACT', 'DRIVCOND', 'MANOEUVER',
                'VEHTYPE', 'INVTYPE', 'Month', 'Hour', 'DayOfWeek', 'INVAGE',
                'PEDESTRIAN', 'CYCLIST', 'AUTOMOBILE', 'MOTORCYCLE', 'TRUCK',
                'PASSENGER', 'SPEEDING', 'AG_DRIV', 'REDLIGHT', 'ALCOHOL',
                'POLICE_DIVISION']
raw_features = [v for v in raw_features if v in ksi.columns]
print(f"  Raw features: {len(raw_features)}", flush=True)

In [ ]:
# --- Quantitative filter: Spearman correlation ---
le_temp = {}
enc_temp = {}
for v in raw_features:
    if ksi[v].dtype in ('object', 'str', 'string') or isinstance(ksi[v].dtype, pd.StringDtype):
        le = LabelEncoder()
        enc_temp[v] = le.fit_transform(ksi[v].astype(str))
    else:
        enc_temp[v] = ksi[v].values
enc_df = pd.DataFrame(enc_temp)

In [ ]:
corr = spearmanr(enc_df.values[np.random.choice(len(enc_df), min(5000, len(enc_df)), replace=False)])
corr_mat = pd.DataFrame(corr[0], index=enc_df.columns, columns=enc_df.columns)
high_corr = set()
for i in range(len(corr_mat.columns)):
    for j in range(i):
        if abs(corr_mat.iloc[i, j]) > 0.85:
            high_corr.add(corr_mat.columns[j])

In [ ]:
features = [v for v in raw_features if v not in high_corr]
if high_corr:
    print(f"  Spearman removed: {high_corr}", flush=True)
print(f"  Final features: {len(features)}  (paper: 26)", flush=True)
print(f"  {features}", flush=True)

In [ ]:
# ====================================================================
# 5. FEATURE MATRIX — One-hot encoding for Lasso sparsity
# ====================================================================
print("\n5. FEATURE MATRIX", flush=True)

In [ ]:
# One-hot encode categoricals
cat_feats = [v for v in features if v in cat_cols]
num_feats = [v for v in features if v not in cat_feats]

In [ ]:
X_parts = []
if num_feats:
    X_num = ksi[num_feats].astype(float)
    X_parts.append(X_num)
if cat_feats:
    X_cat = pd.get_dummies(ksi[cat_feats], prefix_sep='_', drop_first=True).fillna(0).astype(float)
    print(f"  One-hot features: {X_cat.shape[1]} (from {len(cat_feats)} categoricals)", flush=True)
    X_parts.append(X_cat)

In [ ]:
X_all = pd.concat(X_parts, axis=1)
y_all = ksi['target_binary'].values
feature_names = X_all.columns.tolist()
print(f"  X: {X_all.shape[0]:,d} x {X_all.shape[1]} (one-hot encoded)", flush=True)

In [ ]:
# Remove zero-variance columns
vt = VarianceThreshold(threshold=0)
X_all_arr = vt.fit_transform(X_all)
kept_mask = vt.get_support()
feature_names = [f for f, keep in zip(feature_names, kept_mask) if keep]
X_all = pd.DataFrame(X_all_arr, columns=feature_names, index=ksi.index)
print(f"  After variance filter: {X_all.shape[1]} features", flush=True)

In [ ]:
# Store alignment for subgroups: same columns, same order
ohe_feature_names = feature_names

In [ ]:
# ====================================================================
# 6. PER-GROUP SPLITS
# ====================================================================
print("\n6. PER-GROUP SPLITS", flush=True)
group_map = {'Drivers': 'Driver', 'Cyclists': 'Cyclist', 'Pedestrians': 'Pedestrian'}
group_data = {'Collisions': {'X': X_all.values, 'y': y_all, 'df': ksi}}

In [ ]:
for gname, gtype in group_map.items():
    mask = ksi['INVTYPE'] == gtype
    gdf = ksi.loc[mask]
    # Build features matching full set
    gX_parts = []
    if num_feats:
        gX_parts.append(gdf[num_feats].astype(float))
    if cat_feats:
        gX_cat = pd.get_dummies(gdf[cat_feats], prefix_sep='_', drop_first=True).fillna(0).astype(float)
        gX_parts.append(gX_cat)
    gX_raw = pd.concat(gX_parts, axis=1) if gX_parts else pd.DataFrame(index=gdf.index)
    # Align columns to full feature set
    for col in feature_names:
        if col not in gX_raw.columns:
            gX_raw[col] = 0.0
    gX = gX_raw[feature_names].values
    gy = gdf['target_binary'].values
    group_data[gname] = {'X': gX, 'y': gy, 'df': gdf}
    print(f"  {gname}: {len(gy):,d} (pos={gy.sum():,d}, neg={(1-gy).sum():,d})", flush=True)

In [ ]:
# ====================================================================
# 7. APRIORI
# ====================================================================
print("\n7. APRIORI ASSOCIATION RULES", flush=True)

In [ ]:
def mine_rules(df, name):
    print(f"  {name}:", flush=True)
    vars_ = [v for v in ['IMPACTYPE','LIGHT','RDSFCOND','VISIBILITY',
                          'SPEEDING','AG_DRIV','REDLIGHT','ALCOHOL'] if v in df.columns]
    d = df[vars_ + ['target_binary']].head(3000).fillna('Unknown')
    d['tl'] = (d['target_binary'] == 1)
    b = pd.get_dummies(d[vars_], prefix_sep='=').astype(bool)
    b['target=Fatal_Major'] = d['tl'].values
    minc = len(b) * 0.05
    cols = [c for c in b.columns if b[c].sum() >= minc or c == 'target=Fatal_Major']
    b = b[cols]
    try:
        freq = apriori(b, min_support=0.05, use_colnames=True, low_memory=True)
        rules = association_rules(freq, metric='lift', min_threshold=1.0)
        rules = rules[rules['confidence'] >= 0.3].sort_values('lift', ascending=False)
        print(f"    itemsets={len(freq)}, rules={len(rules)}", flush=True)
        for _, r in rules.head(5).iterrows():
            lhs = ', '.join(list(r['antecedents'])[:3])
            rhs = ', '.join(list(r['consequents'])[:2])
            print(f"    {{{lhs}}} => {{{rhs}}}  sup={r['support']:.3f} conf={r['confidence']:.3f} lift={r['lift']:.2f}", flush=True)
    except Exception as e:
        print(f"    error: {e}", flush=True)

In [ ]:
for g in ['Collisions', 'Drivers', 'Cyclists', 'Pedestrians']:
    mine_rules(group_data[g]['df'], g)

In [ ]:
# ====================================================================
# 8. MODELS — Lasso (tuned) + RF, with 10-fold CV
# ====================================================================
print("\n8. MODELS", flush=True)
results = []

In [ ]:
for gname, gd in group_data.items():
    X, y = gd['X'], gd['y']
    print(f"\n  {gname}: {len(y):,d} records (pos={y.sum():,d}, neg={(1-y).sum():,d})", flush=True)
    if len(y) < 50 or y.sum() < 5 or (1-y).sum() < 5:
        print("    SKIP (too few samples for meaningful eval)", flush=True)
        continue

    # 80/20 split (stratified)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
    print(f"    Train: {len(y_tr):,d}, Test: {len(y_te):,d}", flush=True)

    # SMOTE on training only (no data leakage)
    # Paper used SMOTE but not necessarily to full balance;
    # for Pedestrians (94% pos), full balance over-generates synthetic neg.
    pos_train = y_tr.sum()
    neg_train = len(y_tr) - pos_train
    if neg_train / max(pos_train, 1) < 0.2:
        # Extreme imbalance: use ratio=0.5 to avoid over-generating
        sm = SMOTE(random_state=RANDOM_SEED, sampling_strategy=0.5)
        print(f"      Extreme imbalance detected: using ratio=0.5", flush=True)
    else:
        sm = SMOTE(random_state=RANDOM_SEED)
    X_tr_sm, y_tr_sm = sm.fit_resample(X_tr, y_tr)
    print(f"    After SMOTE: {len(y_tr_sm):,d} (pos={y_tr_sm.sum():,d}, neg={(1-y_tr_sm).sum():,d})", flush=True)

    # Scale
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr_sm)
    X_te_sc = sc.transform(X_te)

    # ---- LASSO (C selection with limited search) ----
    print("    Lasso:", flush=True)
    best_C = 1.0
    best_cv = 0
    for C in [0.1, 1.0, 10.0]:
        lr_c = LogisticRegression(penalty='l1', C=C, solver='saga', max_iter=1000,
                                  random_state=RANDOM_SEED, l1_ratio=1)
        scores = cross_val_score(lr_c, X_tr_sc, y_tr_sm, cv=3, scoring='accuracy')
        if scores.mean() > best_cv:
            best_cv = scores.mean()
            best_C = C
    print(f"      Best C={best_C:.4f} (3-fold CV acc={best_cv:.4f})", flush=True)

    lr = LogisticRegression(penalty='l1', C=best_C, solver='saga', max_iter=2000,
                            random_state=RANDOM_SEED, l1_ratio=1)
    lr.fit(X_tr_sc, y_tr_sm)

    # 5-fold CV (paper used 10-fold on 80% train)
    cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    cv_acc = cross_val_score(lr, X_tr_sc, y_tr_sm, cv=cv_outer, scoring='accuracy')
    print(f"      5-fold CV: acc={cv_acc.mean():.4f} +/- {cv_acc.std():.4f}", flush=True)

    # Test
    yp = lr.predict(X_te_sc)
    acc = accuracy_score(y_te, yp)
    sens = recall_score(y_te, yp)
    tn, fp, fn, tp = confusion_matrix(y_te, yp).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    bal_acc = (sens + spec) / 2
    nz = np.sum(np.abs(lr.coef_[0]) > 1e-6)
    print(f"      Test: acc={acc:.4f} sens={sens:.4f} spec={spec:.4f} bal_acc={bal_acc:.4f} nz={nz}", flush=True)
    results.append({'Group': gname, 'Model': 'Lasso',
                    'Accuracy': f'{acc:.4f}', 'Sensitivity': f'{sens:.4f}',
                    'Specificity': f'{spec:.4f}', 'Bal_Acc': f'{bal_acc:.4f}',
                    'CV_Acc': f'{cv_acc.mean():.4f}', 'NZ_Coef': nz})

    # ---- RANDOM FOREST ----
    print("    Random Forest:", flush=True)
    rf = RandomForestClassifier(n_estimators=100, max_depth=10,
                                random_state=RANDOM_SEED, class_weight='balanced')
    rf.fit(X_tr_sc, y_tr_sm)

    cv_acc_rf = cross_val_score(rf, X_tr_sc, y_tr_sm, cv=cv_outer, scoring='accuracy')
    print(f"      5-fold CV: acc={cv_acc_rf.mean():.4f} +/- {cv_acc_rf.std():.4f}", flush=True)

    yp = rf.predict(X_te_sc)
    acc = accuracy_score(y_te, yp)
    sens = recall_score(y_te, yp)
    tn, fp, fn, tp = confusion_matrix(y_te, yp).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    bal_acc = (sens + spec) / 2
    print(f"      Test: acc={acc:.4f} sens={sens:.4f} spec={spec:.4f} bal_acc={bal_acc:.4f}", flush=True)
    results.append({'Group': gname, 'Model': 'RF',
                    'Accuracy': f'{acc:.4f}', 'Sensitivity': f'{sens:.4f}',
                    'Specificity': f'{spec:.4f}', 'Bal_Acc': f'{bal_acc:.4f}',
                    'CV_Acc': f'{cv_acc_rf.mean():.4f}', 'NZ_Coef': '-'})

    # Feature importance
    if hasattr(rf, 'feature_importances_'):
        n_display = min(5, len(feature_names), len(rf.feature_importances_))
        if n_display > 0:
            imp_idx = np.argsort(rf.feature_importances_)[::-1][:n_display]
            print(f"      Top-5 features:", flush=True)
            for idx in imp_idx:
                name = feature_names[idx] if idx < len(feature_names) else f'feat_{idx}'
                print(f"        {name}: {rf.feature_importances_[idx]:.4f}", flush=True)

In [ ]:
# ====================================================================
# 9. SUMMARY vs PAPER
# ====================================================================
print("\n" + "=" * 72)
print("FINAL RESULTS vs PAPER (Shanshal et al., 2020)")
print("=" * 72)
summary = pd.DataFrame(results)
if len(summary) > 0:
    print(summary.to_string(index=False))

In [ ]:
paper_ref = {
    'Drivers':    {'RF': 0.80, 'Lasso': '~0.76-0.84', 'Spec': '~0.68-0.87'},
    'Cyclists':   {'RF': 0.89, 'Lasso': '~0.76-0.84', 'Spec': '~0.4-0.5'},
    'Pedestrians':{'RF': 0.80, 'Lasso': '~0.76-0.84', 'Spec': '~0.4-0.5'},
}
print(f"\n  Paper reference:", flush=True)
for g, v in paper_ref.items():
    print(f"    {g}: RF acc={v['RF']}, Lasso acc={v['Lasso']}, Specificity={v['Spec']}", flush=True)

In [ ]:
summary.to_csv(os.path.join(OUTPUT_DIR, 'results_summary_v3.csv'), index=False)

In [ ]:
# Plot
if len(results) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    labels = [f"{r['Group']}-{r['Model']}" for r in results]
    x = np.arange(len(labels))
    accs = [float(r['Accuracy']) for r in results]
    sens = [float(r['Sensitivity']) for r in results]
    spec = [float(r['Specificity']) for r in results]
    bal = [float(r['Bal_Acc']) for r in results]
    ax.bar(x - 0.3, accs, 0.2, label='Accuracy', color='steelblue')
    ax.bar(x - 0.1, sens, 0.2, label='Sensitivity', color='coral')
    ax.bar(x + 0.1, spec, 0.2, label='Specificity', color='seagreen')
    ax.bar(x + 0.3, bal, 0.2, label='Bal_Acc', color='gold')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylabel('Score')
    ax.set_title('Model Performance — Toronto Paper Reproduction v3')
    ax.legend()
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison_v3.png'), dpi=150)
    plt.close()

In [ ]:
print(f"\nOutputs: {OUTPUT_DIR}/", flush=True)
print("Done.", flush=True)